In [1]:
import os

import numpy as np
import pandas as pd

import torch

/tmp/ipykernel_847931/1882739812.py:4: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


In [2]:
import sys, os
sclembas = '/home/hmbaghda/Projects/scLEMBAS'
sys.path.insert(1, os.path.join(sclembas))
from scLEMBAS import parse_network, io
from scLEMBAS.model.scl import SignalingModel
from scLEMBAS.model.tfa import TFA

In [3]:
n_cores = 12
os.environ["OMP_NUM_THREADS"] = str(n_cores)
os.environ["MKL_NUM_THREADS"] = str(n_cores)
os.environ["OPENBLAS_NUM_THREADS"] = str(n_cores)
os.environ["VECLIB_MAXIMUM_THREADS"] = str(n_cores)
os.environ["NUMEXPR_NUM_THREADS"] = str(n_cores)

seed = 888

device = "cuda" if torch.cuda.is_available() else "cpu"

data_path = '/nobackup/users/hmbaghda/scLEMBAS/analysis'
models_path = os.path.join(data_path, 'processed', 'models')
if not os.path.isdir(models_path):
    os.mkdir(models_path)

/nobackup/users/hmbaghda/Software/miniforge3/envs/scLEMBAS/lib/python3.11/site-packages/torch/cuda/__init__.py:107: UserWarning: CUDA initialization: CUDA unknown error - this may be due to an incorrectly set up environment, e.g. changing env variable CUDA_VISIBLE_DEVICES after program start. Setting the available devices to be zero. (Triggered internally at /opt/conda/conda-bld/pytorch-base_1705959557029/work/c10/cuda/CUDAFunctions.cpp:109.)
  return torch._C._cuda_getDeviceCount() > 0


In [4]:
tf_adata = io.read_tfad(file_name = os.path.join(data_path, 'processed', 'ID_tf_activity.h5ad'))

In [5]:
source_label = 'source_genesymbol'
target_label = 'target_genesymbol'
weight_label = 'mode_of_action'
stimulation_label = 'consensus_stimulation'
inhibition_label = 'consensus_inhibition'

sn_ppis = parse_network.load_network('omnipath', organism = 'mouse', static = True)
sn_ppis = parse_network.extract_network(sn_ppis, curation_effort_thresh = 5, n_references_thresh = 3,
                                        resources = ['HuRI','IntAct','KEGG-MEDICUS','NetPath','Reactome_SignaLink3','SPIKE','SignaLink3','SIGNOR', 
                                                'Baccin2019', 'Ramilowski2015', 'Reactome_LRdb', 'UniProt_LRdb', 'CellChatDB', 'CellPhoneDB', 'connectomeDB2020', 'scConnect'], 
                                        drop_self = True, verbose = True)

tf_labels = tf_adata.var.index.unique().tolist()

ligand_labels = tf_adata.obs['sample'].unique().tolist()
ligand_labels = [(l[0] + l[1:].lower()).replace('-', '') for l in ligand_labels] # mouse naming convention

# filter for paths b/w ligand and tf
fn_1 = parse_network.fully_connected_network(sn_ppis, ligand_labels, tf_labels, source_label = source_label, target_label = target_label, 
                       path_finder = 'shortest')
fn_2 = parse_network.fully_connected_network(sn_ppis, ligand_labels, tf_labels, source_label = source_label, target_label = target_label, 
                       path_finder = 'connected')
# of the methods to identify paths, retain the one that has the most interactions
if fn_1.shape[0] > fn_2.shape[0]:
    sn_ppis = fn_1
else:
    sn_ppis = fn_2

del fn_1, fn_2

sn_ppis.loc[sn_ppis[(sn_ppis[stimulation_label] == 1) & (sn_ppis[inhibition_label] == 1)].index, 
    [stimulation_label, inhibition_label]] = [False, False]
sn_ppis = parse_network.format_network(sn_ppis, weight_label, stimulation_label, inhibition_label)

all_nodes = sn_ppis[source_label].tolist() + sn_ppis[target_label].tolist()
input_ligands_available = sorted(set(ligand_labels).intersection(all_nodes))

The thresholds filtered 21403  of 28277 interactions
The resources filtered 937  of 6874 interactions


100%|████████████████████████████████████| 8122/8122 [00:00<00:00, 21370.02it/s]


In [6]:
subset_tf = tf_adata[tf_adata.obs.TF_clusters.isin(['9', '15'])]
sample_size = subset_tf.obs.TF_clusters.value_counts().min()
large_cluster = subset_tf.obs.TF_clusters.value_counts().idxmax()
small_cluster = subset_tf.obs.TF_clusters.value_counts().idxmin()
large_cluster_barcodes = subset_tf.obs[subset_tf.obs.TF_clusters == large_cluster].index
small_cluster_barcodes = subset_tf.obs[subset_tf.obs.TF_clusters == small_cluster].index.tolist()
np.random.seed(seed)
lcb_sub = list(np.random.choice(large_cluster_barcodes, sample_size, replace = False))
subset_tf = subset_tf[lcb_sub + small_cluster_barcodes, :]
subset_tf.obs.TF_clusters.value_counts()

np.random.seed(seed)
selected_ligand = np.random.choice(input_ligands_available, 1)[0]

ligand_input = pd.DataFrame(subset_tf.obs.TF_clusters.map({'9': 0, '15': 1}))
ligand_input.columns = [selected_ligand]
tf_output = pd.DataFrame(subset_tf.X, index = subset_tf.obs.index, columns = subset_tf.var.index)

In [7]:
# linear scaling of inputs/outputs
projection_amplitude_in = 3
projection_amplitude_out = 1.2
# other parameters
bionet_params = {'target_steps': 100, 'max_steps': 120, 'exp_factor':50, 'tolerance': 1e-5, 'leak':1e-2} # fed directly to model

# training parameters
lr_params = {'max_iter': 5000, 
             'learning_rate': 2e-3}
other_params = {'batch_size': 256, 'noise_level': 10, 'gradient_noise_level': 1e-9}
regularization_params = {'param_lambda_L2': 1e-6, 'moa_lambda_L1': 0.1, 'ligand_lambda_L2': 1e-5, 'uniform_lambda_L2': 1e-4, 
                   'uniform_max': 1/projection_amplitude_out, 'spectral_loss_factor': 1e-5}
spectral_radius_params = {'n_probes_spectral': 5, 'power_steps_spectral': 50, 'subset_n_spectral': 10}
target_spectral_radius = 0.8
hyper_params = {**lr_params, **other_params, **regularization_params, **spectral_radius_params}


Hyper_params = {
    'tfa': {'n_latent': 32, 'cat_max_norm': 1, 'recon_loss': 'gauss'}, 
    'encoder': {'n_latent': 64, 'n_hidden_layers': 1, 'n_hidden_nodes': [256], 'batch_momentum': 0.01, 
                'layer_norm': False, 'dropout_rate': 0.1, 'activation_fn': torch.nn.ReLU}
}

In [31]:
mod = SignalingModel(net = sn_ppis,
                     X_in = ligand_input,
                     y_out = tf_output, 
                     activation_function = 'MML', 
                     skip_bionet_out = False,
                     covariates = subset_tf.obs,
                     categorical_covariate_keys = ['TF_clusters', 'celltype'],
                     projection_amplitude_in = projection_amplitude_in, projection_amplitude_out = projection_amplitude_out,
                     weight_label = weight_label, source_label = source_label, target_label = target_label,
                     bionet_params = bionet_params, 
                     tfa_hyper_params = Hyper_params['tfa'], 
                     encoder_hyper_params = Hyper_params['encoder'], 
                     dtype = torch.float32, device = device, seed = seed)

In [32]:
mod.tf_autoencoder

TFA(
  (encoder): Encoder(
    (encoder): Sequential(
      (Encoder Layer 0): Sequential(
        (linear): Linear(in_features=193, out_features=127, bias=True)
        (batch normalization): BatchNorm1d(127, eps=1e-05, momentum=0.01, affine=True, track_running_stats=True)
        (activation): ReLU()
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (Encoder Layer 1): Sequential(
        (linear): Linear(in_features=127, out_features=33, bias=True)
        (batch normalization): BatchNorm1d(33, eps=1e-05, momentum=0.01, affine=True, track_running_stats=True)
        (activation): ReLU()
        (dropout): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (cat_embeddings): ModuleDict(
    (TF_clusters): Embedding(2, 33)
    (celltype): Embedding(12, 33)
  )
)

In [9]:
self = mod
X_in = mod.df_to_tensor(mod.X_in)

In [10]:
X_full = self.input_layer(X_in) # input ligands to signaling network
Y_full = self.signaling_network(X_full) # RNN of full signaling network
Y_hat = self.output_layer(Y_full) # TF outputs of signaling network
z_basal = self.tf_autoencoder(Y_hat)

In [16]:
mod.tf_autoencoder

TFA(
  (encoder): Encoder(
    (encoder): Sequential(
      (Encoder Layer 0): Sequential(
        (linear): Linear(in_features=193, out_features=256, bias=True)
        (batch normalization): BatchNorm1d(256, eps=1e-05, momentum=0.01, affine=True, track_running_stats=True)
        (activation): ReLU()
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (Encoder Layer 1): Sequential(
        (linear): Linear(in_features=256, out_features=32, bias=True)
        (batch normalization): BatchNorm1d(32, eps=1e-05, momentum=0.01, affine=True, track_running_stats=True)
        (activation): ReLU()
        (dropout): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (cat_embeddings): ModuleDict(
    (TF_clusters): Embedding(2, 32, max_norm=1)
    (celltype): Embedding(12, 32, max_norm=1)
  )
)

In [15]:
z_basal

tensor([[0.4603, 1.0695, 0.0000,  ..., 0.0000, 0.0000, 0.7087],
        [0.5586, 1.6296, 0.0000,  ..., 0.0000, 0.0000, 0.0654],
        [0.7898, 0.6660, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        ...,
        [0.0000, 0.0000, 0.7904,  ..., 0.5903, 1.0020, 0.0000],
        [0.0000, 0.0000, 1.0630,  ..., 1.1239, 0.7841, 0.0000],
        [0.0000, 0.0000, 0.0000,  ..., 0.2900, 0.1050, 0.0000]],
       grad_fn=<MulBackward0>)

In [14]:
mod2 = SignalingModel(net = sn_ppis,
                     X_in = ligand_input,
                     y_out = tf_output, 
                     activation_function = 'MML', 
                     skip_bionet_out = False,
                     covariates = subset_tf.obs,
                     categorical_covariate_keys = ['TF_clusters', 'celltype'],
                     projection_amplitude_in = projection_amplitude_in, projection_amplitude_out = projection_amplitude_out,
                     weight_label = weight_label, source_label = source_label, target_label = target_label,
                     bionet_params = bionet_params, 
                     tfa_hyper_params = Hyper_params['tfa'], 
                     encoder_hyper_params = Hyper_params['encoder'], 
                     dtype = torch.float32, device = device, seed = seed)

self = mod2
X_in = mod2.df_to_tensor(mod.X_in)

X_full = self.input_layer(X_in) # input ligands to signaling network
Y_full = self.signaling_network(X_full) # RNN of full signaling network
Y_hat = self.output_layer(Y_full) # TF outputs of signaling network
z_basal = self.tf_autoencoder(Y_hat)

torch.Size([2336, 193])

In [45]:
Y_full[:, self.output_node_order]

tensor([[0.0023, 0.1455, 0.0019,  ..., 0.0013, 0.7457, 0.0012],
        [0.0023, 0.1455, 0.0019,  ..., 0.0013, 0.7457, 0.0012],
        [0.0023, 0.1455, 0.0019,  ..., 0.0013, 0.7457, 0.0012],
        ...,
        [0.0028, 0.1455, 0.0020,  ..., 0.0014, 0.7457, 0.0013],
        [0.0028, 0.1455, 0.0020,  ..., 0.0014, 0.7457, 0.0013],
        [0.0028, 0.1455, 0.0020,  ..., 0.0014, 0.7457, 0.0013]],
       grad_fn=<IndexBackward0>)

In [46]:
self.weights * Y_full[:, self.output_node_order]

tensor([[0.0027, 0.1746, 0.0022,  ..., 0.0016, 0.8949, 0.0015],
        [0.0027, 0.1746, 0.0022,  ..., 0.0016, 0.8949, 0.0015],
        [0.0027, 0.1746, 0.0022,  ..., 0.0016, 0.8949, 0.0015],
        ...,
        [0.0034, 0.1746, 0.0024,  ..., 0.0017, 0.8949, 0.0015],
        [0.0034, 0.1746, 0.0024,  ..., 0.0017, 0.8949, 0.0015],
        [0.0034, 0.1746, 0.0024,  ..., 0.0017, 0.8949, 0.0015]],
       grad_fn=<MulBackward0>)

In [49]:
self.weights[1]

tensor(1.2000, grad_fn=<SelectBackward0>)

In [50]:
self.weights[1]*0.1455

tensor(0.1746, grad_fn=<MulBackward0>)

In [13]:
X_in.shape

torch.Size([2336, 1])

In [14]:
X_full.shape # samples x n_nodes in network

torch.Size([2336, 332])

In [30]:
Y_full.shape

torch.Size([2336, 332])

In [33]:
Y_hat.shape

torch.Size([2336, 193])